# MSCM-net: Salinas experiments
Runtime must be **GPU** (Runtime → Change runtime type → T4).

Order: Cell 1 → Cell 2 → Cell 3 (download) → Cell 4 (data check) → Cell 5 (baseline) → Cell 6 (augmentation).
After any reconnect, run Cell 1 and Cell 2 again. Finished seeds are saved on Drive and skipped.

## Cell 1: mount Drive and check code files

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
sys.path.append('/content/drive/MyDrive/MSCMnet/codes')
print(sorted(os.listdir('/content/drive/MyDrive/MSCMnet/codes')))
# expect: experiments.py, hsi_data.py, mscm_modules.py, mscm_net.py, train.py

## Cell 2: load code, set Salinas paths

In [ ]:
import sys, importlib, glob, torch
import numpy as np
sys.path.append('/content/drive/MyDrive/MSCMnet/codes')
sys.path_importer_cache.clear()

import hsi_data, mscm_modules, mscm_net, train, experiments
for m in (hsi_data, mscm_modules, mscm_net, train, experiments):
    importlib.reload(m)

R = '/content/drive/MyDrive/MSCMnet'
S = f'{R}/data/Salinas'
os.makedirs(S, exist_ok=True)
sal_data, sal_gt = f'{S}/Salinas_corrected.mat', f'{S}/Salinas_gt.mat'

print('GPU available:', torch.cuda.is_available())
print('files present:', os.path.exists(sal_data), os.path.exists(sal_gt))

## Cell 3: download Salinas into Drive
Colab downloads from Google's servers, so a block on your own connection may not apply. If a file is already on Drive it is skipped.

In [ ]:
import urllib.request

def is_mat(path):
    # real MATLAB files start with b'MATLAB'; an error web page starts with '<'
    try:
        with open(path, 'rb') as f:
            return f.read(6) == b'MATLAB'
    except OSError:
        return False

sources = {
    'Salinas_corrected.mat': ['http://www.ehu.eus/ccwintco/uploads/a/a3/Salinas_corrected.mat',
                              'https://www.ehu.eus/ccwintco/uploads/a/a3/Salinas_corrected.mat'],
    'Salinas_gt.mat':        ['http://www.ehu.eus/ccwintco/uploads/f/fa/Salinas_gt.mat',
                              'https://www.ehu.eus/ccwintco/uploads/f/fa/Salinas_gt.mat'],
}

for name, urls in sources.items():
    dest = f'{S}/{name}'
    if is_mat(dest):
        print(name, 'already on Drive'); continue
    for u in urls:
        try:
            req = urllib.request.Request(u, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=120) as r, open(dest, 'wb') as f:
                f.write(r.read())
            if is_mat(dest):
                print('downloaded', name, os.path.getsize(dest) // 1024, 'KB'); break
            print('not a MATLAB file from', u); os.remove(dest)
        except Exception as e:
            print('failed', u, '->', repr(e)[:120])
    else:
        print('COULD NOT DOWNLOAD', name, '-> use the fallback cell below')

## Cell 3b (only if Cell 3 failed): look for Salinas on the Hugging Face repo you used before
This only lists matching files. Send me the printed list and I will give you the exact download lines.

In [ ]:
from huggingface_hub import HfApi
files = HfApi().list_repo_files('Tanishq165/HSI_Datasets', repo_type='dataset')
print([f for f in files if 'salinas' in f.lower()])
print(len(files), 'files in repo; first 30:', files[:30])

## Cell 3c (only if everything else failed): upload the two files from your computer
Get them any other way (another network, phone, a mirror), then upload both `.mat` files here.

In [ ]:
from google.colab import files as colab_files
up = colab_files.upload()          # choose Salinas_corrected.mat and Salinas_gt.mat
import shutil
for name in up:
    shutil.move(name, f'{S}/{name}')
print(os.listdir(S))

## Cell 4: check the data against the paper's Table 1

In [ ]:
from hsi_data import load_hsi

img, g = load_hsi(sal_data, sal_gt)
print('image', img.shape, ' gt', g.shape)          # expect (512, 217, 204) and (512, 217)
print('classes', np.unique(g))                     # expect 0..16
print('labeled pixels', int((g > 0).sum()))        # expect 54129

expected = [2009, 3726, 1976, 1394, 2678, 3959, 3579, 11271, 6203, 3278, 1068, 1927, 916, 1070, 7268, 1807]
counts = np.bincount(g.ravel())[1:].tolist()
print('class counts match paper Table 1:', counts == expected)
if counts != expected:
    print(counts)

## Cell 5: baseline (faithful to the paper, no augmentation), 10 seeds
About 30 to 50 minutes. Keep the tab open.

In [ ]:
experiments.run_config('sal_base', sal_data, sal_gt, R, cache_name='salinas')

## Cell 6: augmentation, 10 seeds, then paired comparison

In [ ]:
experiments.run_config('sal_aug', sal_data, sal_gt, R, cache_name='salinas', augment=True)
experiments.compare(R, 'sal_aug', baseline='sal_base')

## Summary of saved Salinas results (no training)

In [ ]:
def summary(tag):
    rs = [np.load(f) for f in sorted(glob.glob(f'{R}/results/{tag}_seed*.npz'))]
    for k in ('OA', 'AA', 'Kappa'):
        v = np.array([float(r[k]) for r in rs])
        print(f'{tag:9s} {k:6s} {v.mean():.2f} ± {v.std():.2f}   (min {v.min():.2f}, max {v.max():.2f}, n={len(v)})')

for tag in ('sal_base', 'sal_aug'):
    if glob.glob(f'{R}/results/{tag}_seed*.npz'):
        summary(tag)
print('paper: OA 98.35  AA 98.99  Kappa 98.16')